# Option Greeks via Finite Differences

The Crank-Nicolson FD solver returns the entire price surface $V(S, t=0)$ as a vector indexed by stock price. Because that surface is a discrete object, the spatial Greeks (Delta and Gamma) are simply central differences on the stored array, and require **no extra solves**.

Theta is recovered by solving on a slightly shorter time horizon and finite differencing across $t$. Vega and Rho do not have a directly accessible derivative direction in the grid, so we use the standard bump-and-reprice approach.

We compare every Greek against the closed-form Black-Scholes formulas to verify accuracy.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from finite_difference import (
    OptionParams, GridParams, bs_price,
    solve_crank_nicolson, make_stock_grid
)

## Parameters

In [ ]:
S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
option_type = "put"
S_max, M, N = 300.0, 300, 300

## Delta and Gamma

Both are spatial derivatives of the price surface and come for free from the FD grid:

$$\Delta_i \approx \frac{V_{i+1} - V_{i-1}}{S_{i+1} - S_{i-1}}, \qquad \Gamma_i \approx \frac{V_{i+1} - 2 V_i + V_{i-1}}{(\Delta S)^2}$$

We use `np.gradient` which applies central differences in the interior and one-sided differences at the edges.

In [ ]:
S, V = solve_crank_nicolson(S_max, K, r, sigma, T, M, N, option_type=option_type)
dS = S[1] - S[0]

delta = np.gradient(V, S)
gamma = np.gradient(delta, S)

idx = int(np.argmin(np.abs(S - S0)))
print(f"S nearest S0: {S[idx]:.4f}")
print(f"FD Delta at S0: {delta[idx]:.6f}")
print(f"FD Gamma at S0: {gamma[idx]:.6f}")

In [ ]:
def bs_delta(S, K, T, r, sigma, option_type="call"):
    """Closed-form Black-Scholes Delta."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    if option_type == "call":
        return norm.cdf(d1)
    return norm.cdf(d1) - 1.0

def bs_gamma(S, K, T, r, sigma):
    """Closed-form Black-Scholes Gamma (same for call and put)."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return norm.pdf(d1) / (S * sigma * np.sqrt(T))

In [ ]:
delta_bs = bs_delta(S0, K, T, r, sigma, option_type)
gamma_bs = bs_gamma(S0, K, T, r, sigma)

print(f"{'':10s} {'FD':>12s} {'Analytical':>12s} {'Abs error':>12s}")
print(f"{'Delta':10s} {delta[idx]:12.6f} {delta_bs:12.6f} {abs(delta[idx] - delta_bs):12.2e}")
print(f"{'Gamma':10s} {gamma[idx]:12.6f} {gamma_bs:12.6f} {abs(gamma[idx] - gamma_bs):12.2e}")

In [ ]:
mask = (S >= 40) & (S <= 180)
S_plot = S[mask]
delta_bs_plot = bs_delta(S_plot, K, T, r, sigma, option_type)
gamma_bs_plot = bs_gamma(S_plot, K, T, r, sigma)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(S_plot, delta[mask], label='FD Delta', lw=2)
axes[0].plot(S_plot, delta_bs_plot, '--', label='Analytical Delta', lw=1.5)
axes[0].axvline(K, color='k', ls=':', alpha=0.5, label='Strike')
axes[0].set_xlabel('Stock price S')
axes[0].set_ylabel('Delta')
axes[0].set_title(f'Delta of European {option_type.title()}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(S_plot, gamma[mask], label='FD Gamma', lw=2)
axes[1].plot(S_plot, gamma_bs_plot, '--', label='Analytical Gamma', lw=1.5)
axes[1].axvline(K, color='k', ls=':', alpha=0.5, label='Strike')
axes[1].set_xlabel('Stock price S')
axes[1].set_ylabel('Gamma')
axes[1].set_title(f'Gamma of European {option_type.title()}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Theta

Theta is the partial derivative of the option price with respect to calendar time: $\Theta = \partial V / \partial t$. We obtain it numerically by solving the PDE for two horizons separated by a small $\Delta t$, and finite differencing.

By convention long options have **negative Theta** (value decays as time passes). Per-day Theta is the total Theta divided by 365.

In [ ]:
dt_bump = 1.0 / 252.0

S_T,  V_T  = solve_crank_nicolson(S_max, K, r, sigma, T,           M, N, option_type=option_type)
S_Tm, V_Tm = solve_crank_nicolson(S_max, K, r, sigma, T - dt_bump, M, N, option_type=option_type)

idx_T  = int(np.argmin(np.abs(S_T  - S0)))
idx_Tm = int(np.argmin(np.abs(S_Tm - S0)))

# As calendar time advances by dt_bump, time-to-expiry shrinks; the option with shorter T
# represents the value at t = dt_bump. Theta = (V(t+dt) - V(t)) / dt.
theta_fd = (V_Tm[idx_Tm] - V_T[idx_T]) / dt_bump

theta_bs = (
    bs_price(S0, K, T - dt_bump, r, sigma, option_type)
    - bs_price(S0, K, T,         r, sigma, option_type)
) / dt_bump

print(f"FD Theta (per year):     {theta_fd:.6f}")
print(f"Analytical Theta (yr):   {theta_bs:.6f}")
print(f"FD Theta (per day):      {theta_fd / 365:.6f}")
print(f"Analytical Theta (day):  {theta_bs / 365:.6f}")

In [ ]:
theta_curve_fd = (V_Tm - V_T) / dt_bump
theta_curve_bs = np.array([
    (bs_price(s, K, T - dt_bump, r, sigma, option_type)
     - bs_price(s, K, T,         r, sigma, option_type)) / dt_bump
    for s in S_T
])

mask = (S_T >= 40) & (S_T <= 180)
plt.figure(figsize=(10, 6))
plt.plot(S_T[mask], theta_curve_fd[mask], label='FD Theta', lw=2)
plt.plot(S_T[mask], theta_curve_bs[mask], '--', label='Analytical Theta', lw=1.5)
plt.axvline(K, color='k', ls=':', alpha=0.5, label='Strike')
plt.xlabel('Stock price S')
plt.ylabel('Theta (per year)')
plt.title(f'Theta of European {option_type.title()}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Vega

Vega is the sensitivity of the price to volatility, $\partial V / \partial \sigma$. There is no $\sigma$ axis in the grid, so we **bump and reprice**: solve at $\sigma$ and $\sigma + \Delta \sigma$, then forward-difference.

In [ ]:
d_sigma = 0.01

_,    V_base = solve_crank_nicolson(S_max, K, r, sigma,            T, M, N, option_type=option_type)
_,    V_up   = solve_crank_nicolson(S_max, K, r, sigma + d_sigma,  T, M, N, option_type=option_type)

vega_fd = (V_up[idx] - V_base[idx]) / d_sigma

d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
vega_bs = S0 * norm.pdf(d1) * np.sqrt(T)

print(f"FD Vega:          {vega_fd:.6f}")
print(f"Analytical Vega:  {vega_bs:.6f}")
print(f"Per 1-vol-pt FD:  {vega_fd / 100:.6f}")

## Rho

Rho is the sensitivity to the risk-free rate, $\partial V / \partial r$. Again we bump-and-reprice.

In [ ]:
d_r = 0.001

_, V_r_base = solve_crank_nicolson(S_max, K, r,       sigma, T, M, N, option_type=option_type)
_, V_r_up   = solve_crank_nicolson(S_max, K, r + d_r, sigma, T, M, N, option_type=option_type)

rho_fd = (V_r_up[idx] - V_r_base[idx]) / d_r

d2 = d1 - sigma * np.sqrt(T)
if option_type == "put":
    rho_bs = -K * T * np.exp(-r * T) * norm.cdf(-d2)
else:
    rho_bs =  K * T * np.exp(-r * T) * norm.cdf( d2)

print(f"FD Rho:           {rho_fd:.6f}")
print(f"Analytical Rho:   {rho_bs:.6f}")

## Greeks Summary Table

In [ ]:
rows = [
    ("Delta", delta[idx], delta_bs),
    ("Gamma", gamma[idx], gamma_bs),
    ("Theta", theta_fd,    theta_bs),
    ("Vega",  vega_fd,     vega_bs),
    ("Rho",   rho_fd,      rho_bs),
]

print(f"{'Greek':<8s} {'FD Value':>14s} {'Analytical':>14s} {'Abs error':>14s}")
print("-" * 54)
for name, fd_val, an_val in rows:
    print(f"{name:<8s} {fd_val:>14.6f} {an_val:>14.6f} {abs(fd_val - an_val):>14.2e}")

## Gamma Near Expiry - Numerical Instability

The terminal payoff has a kink at $S = K$. As $T \to 0$ the analytical Gamma diverges to a delta function at the strike, so any FD scheme will produce a sharp spike there. The discrete Gamma cannot exceed roughly $1/(\Delta S)$ in magnitude, which understates the true value, and Crank-Nicolson can also exhibit small oscillations around the strike from the non-smooth initial data.

This is a well-known FD challenge near expiry. Mitigations include local grid refinement, payoff smoothing (Rannacher start-up), or using a more robust scheme (fully implicit) for the first few time steps.

In [ ]:
T_short = 0.05

S_short, V_short = solve_crank_nicolson(S_max, K, r, sigma, T_short, M, N, option_type=option_type)
delta_short = np.gradient(V_short, S_short)
gamma_short = np.gradient(delta_short, S_short)
gamma_short_bs = bs_gamma(S_short, K, T_short, r, sigma)

mask = (S_short >= 80) & (S_short <= 120)
plt.figure(figsize=(10, 6))
plt.plot(S_short[mask], gamma_short[mask],    label=f'FD Gamma (T={T_short})', lw=2)
plt.plot(S_short[mask], gamma_short_bs[mask], '--', label='Analytical Gamma', lw=1.5)
plt.axvline(K, color='k', ls=':', alpha=0.5, label='Strike')
plt.xlabel('Stock price S')
plt.ylabel('Gamma')
plt.title('Gamma Spike Near Expiry (T = 0.05)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()